# Laser Cutting Intelligence: Surface Roughness Prediction via Ridge Regression

---
**Project 10 · LozanoLsa Industrial ML Portfolio**  
**Domain:** Precision Fabrication — CNC Laser Cutting Process Engineering  
**Algorithm:** Ridge Regression (L2 regularisation + RidgeCV for alpha selection)  
**Target:** `surface_roughness_ra_um` — Surface roughness Ra (µm, continuous)

---

## 🏭 Business Context

In precision laser cutting, surface roughness is the primary quality gate. A part with Ra above specification must be deburred, polished, or scrapped — all of which cost time and money that cannot be recovered in high-mix, low-batch environments. The challenge is not a lack of process data; modern CNC laser systems log dozens of parameters at every cut. The challenge is that the most important process variables are **structurally correlated**: laser power, cutting speed, and assist gas pressure are not set independently — an operator who increases power typically increases speed and gas pressure to match. This creates multicollinearity that inflates OLS coefficient estimates and makes them unreliable for process guidance.

**Ridge regression is the right tool for this problem.** The L2 penalty shrinks all coefficients toward zero proportionally, stabilising estimates when features are correlated without discarding any of them. Unlike Lasso (which zeroes features), Ridge keeps all variables in play — appropriate here because every parameter in the laser cutting recipe has a physical justification for inclusion.

This project:
1. Documents the multicollinearity structure (power–speed–pressure triplet, VIF up to 7.6)
2. Shows Ridge coefficient stability vs OLS instability via regularisation path analysis
3. Delivers a recipe simulator: given any material/thickness combination, predict Ra and benchmark against specification

---

*"When your process variables move together, your model needs a steadying hand — that's what Ridge does."*


## 1 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy import stats
from scipy.stats import normaltest, probplot

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, RidgeCV, LinearRegression
from sklearn.linear_model import ridge_regression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_goldfeldquandt

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

C_BLUE  = "#4C9BE8"
C_RED   = "#E8574C"
C_GREEN = "#2ECC71"
C_AMBER = "#F39C12"
C_DARK  = "#1B2840"
C_MUTED = "#697888"
C_PURPLE= "#9B59B6"

FEATURES = [
    "laser_power_w",
    "cutting_speed_mm_s",
    "assist_gas_flow_l_min",
    "focal_offset_mm",
    "material_thickness_mm",
    "material_type",
    "oxygen_pct",
    "shop_temp_c",
]
TARGET = "surface_roughness_ra_um"

# Roughness specification threshold (example: Ra ≤ 3.2 µm for structural cuts)
RA_SPEC = 3.2

print(f"✓ Environment ready — {len(FEATURES)} process variables")
print(f"  Ra specification limit: ≤ {RA_SPEC} µm")


## 2 · Load Data

The dataset contains around 1,500 cut records extracted from the CNC laser system controller logs, complemented by surface quality measurements from the post-cut CMM inspection station. Material type (0 = carbon steel, 1 = aluminium) was recorded as a binary flag at job setup.

| Column | Type | Description |
|---|---|---|
| `laser_power_w` | float | Laser beam power (W) |
| `cutting_speed_mm_s` | float | Torch travel speed (mm/s) |
| `assist_gas_flow_l_min` | float | Assist gas flow rate (L/min) |
| `focal_offset_mm` | float | Focal point offset from surface (mm; 0 = ideal) |
| `material_thickness_mm` | float | Sheet thickness (mm) |
| `material_type` | int | 0 = Carbon steel · 1 = Aluminium |
| `oxygen_pct` | float | Oxygen content in assist gas (%) |
| `shop_temp_c` | float | Ambient temperature in cutting bay (°C) |
| `surface_roughness_ra_um` | float | **Target** — Ra surface roughness (µm) |


In [ ]:
try:
    df = pd.read_csv("laser_cutting_data.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/LozanoLsa/Ridge_Stable_Process_Modeling/main/laser_cutting_data.csv")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Material mix: {(df['material_type']==0).sum()} steel  |  {(df['material_type']==1).sum()} aluminium")
df.head()


## 3 · Sanity Checks

In [ ]:
print("── Shape ────────────────────────────")
print(f"  Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

print("\n── Data Types ──────────────────────")
print(df.dtypes.to_string())

print("\n── Descriptive Statistics ──────────")
display(df.describe().round(3).T)


In [ ]:
print("── Missing Values ──────────────────")
nulls = df.isna().sum()
print(nulls[nulls > 0] if nulls.any() else "  ✓ No missing values.")

print("\n── Duplicate Rows ──────────────────")
dups = df.duplicated().sum()
print(f"  {dups} duplicates." if dups else "  ✓ No duplicates.")

print("\n── Target Summary ──────────────────")
print(f"  surface_roughness_ra_um: [{df[TARGET].min():.3f}, {df[TARGET].max():.3f}] µm")
print(f"  Mean ± Std: {df[TARGET].mean():.3f} ± {df[TARGET].std():.3f} µm")
pct_ok = (df[TARGET] <= RA_SPEC).mean() * 100
print(f"  Within spec (Ra ≤ {RA_SPEC} µm): {pct_ok:.1f}% of cuts")
print(f"\n  Carbon steel Ra : {df[df['material_type']==0][TARGET].mean():.3f} µm (mean)")
print(f"  Aluminium Ra    : {df[df['material_type']==1][TARGET].mean():.3f} µm (mean)")


## 4 · Exploratory Data Analysis

Ridge regression is motivated by **multicollinearity** — the structural correlation between process variables. The EDA here serves a dual purpose: understand the roughness landscape, and **diagnose the collinearity problem** that makes plain OLS unreliable. The correlation matrix and VIF scores are not just diagnostic tools — they are the business justification for choosing Ridge over OLS.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── (A) Roughness distribution
ax = axes[0]
steel = df[df["material_type"] == 0][TARGET]
alum  = df[df["material_type"] == 1][TARGET]
ax.hist(steel, bins=35, alpha=0.65, color=C_BLUE, label=f"Steel (n={len(steel):,})", edgecolor="white", lw=0.3)
ax.hist(alum,  bins=35, alpha=0.65, color=C_AMBER, label=f"Aluminium (n={len(alum):,})", edgecolor="white", lw=0.3)
ax.axvline(RA_SPEC, color=C_RED, ls="--", lw=1.8, label=f"Spec limit {RA_SPEC} µm")
ax.set_xlabel("Ra Surface Roughness (µm)", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_title("Roughness Distribution by Material", fontsize=11, fontweight="bold")
ax.legend(fontsize=8)

# ── (B) Thickness vs Ra
ax2 = axes[1]
ax2.scatter(df["material_thickness_mm"], df[TARGET],
            c=df["material_type"].map({0: C_BLUE, 1: C_AMBER}),
            alpha=0.25, s=8)
m, b, r, *_ = stats.linregress(df["material_thickness_mm"], df[TARGET])
xr = np.linspace(1, 15, 100)
ax2.plot(xr, m * xr + b, color=C_RED, lw=1.5, ls="--", label=f"r = {r:.3f}")
ax2.axhline(RA_SPEC, color=C_RED, ls=":", lw=1.2, alpha=0.5)
ax2.set_xlabel("Material Thickness (mm)", fontsize=10)
ax2.set_ylabel("Surface Roughness Ra (µm)", fontsize=10)
ax2.set_title("Thickness vs Roughness\n(blue = steel · amber = aluminium)", fontsize=11, fontweight="bold")
ax2.legend(fontsize=8)

# ── (C) Power vs Speed — multicollinearity exhibit #1
ax3 = axes[2]
sc = ax3.scatter(df["laser_power_w"], df["cutting_speed_mm_s"],
                 c=df[TARGET], cmap="RdYlGn_r", alpha=0.35, s=8)
plt.colorbar(sc, ax=ax3, label="Ra (µm)")
ax3.set_xlabel("Laser Power (W)", fontsize=10)
ax3.set_ylabel("Cutting Speed (mm/s)", fontsize=10)
ax3.set_title(f"Power vs Speed (r = 0.907)\nThe multicollinearity problem",
              fontsize=11, fontweight="bold")

plt.suptitle("Target Overview & Multicollinearity Context", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Multicollinearity — this is the Ridge motivation ──────────────────

# Pearson correlation matrix — features only
corr_f = df[FEATURES].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
ax = axes[0]
mask = np.triu(np.ones_like(corr_f, dtype=bool), k=1)
sns.heatmap(corr_f, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn_r",
            center=0, linewidths=0.5, linecolor="white",
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Feature Correlation Matrix\nRed = strong positive correlation → multicollinearity risk",
             fontsize=11, fontweight="bold")

# VIF bar chart
ax2 = axes[1]
X_vif = sm.add_constant(df[FEATURES])
vif_df = pd.DataFrame({
    "Feature": FEATURES,
    "VIF"    : [variance_inflation_factor(X_vif.values, i + 1) for i in range(len(FEATURES))]
}).sort_values("VIF", ascending=True)
bar_c = [C_RED if v > 5 else C_AMBER if v > 2 else C_GREEN for v in vif_df["VIF"]]
bars = ax2.barh(vif_df["Feature"], vif_df["VIF"], color=bar_c, alpha=0.80, edgecolor="none")
ax2.axvline(5, color=C_RED, lw=1.2, ls="--", label="VIF = 5 (moderate concern)")
ax2.axvline(2, color=C_AMBER, lw=1.0, ls=":", label="VIF = 2 (mild concern)")
for bar, val in zip(bars, vif_df["VIF"]):
    ax2.text(val + 0.1, bar.get_y() + bar.get_height() / 2,
             f"{val:.2f}", va="center", fontsize=9)
ax2.set_xlabel("Variance Inflation Factor (VIF)", fontsize=10)
ax2.set_title("VIF — Multicollinearity Severity\nRed bars = Ridge is justified",
              fontsize=11, fontweight="bold")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("High-VIF features (Ridge territory):")
for _, row in vif_df[vif_df["VIF"] > 2].sort_values("VIF", ascending=False).iterrows():
    flag = "⚠ Ridge recommended" if row["VIF"] > 5 else "◉ Moderate concern"
    print(f"  {row['Feature']:<26}: VIF = {row['VIF']:.2f}  {flag}")
print("\nKey collinear triplet: laser_power_w ↔ cutting_speed_mm_s ↔ assist_gas_flow_l_min")
print("  These three move together in laser cutting practice (higher power → higher speed + gas)")


In [ ]:
top_feats = ["material_thickness_mm", "laser_power_w",
             "assist_gas_flow_l_min", "cutting_speed_mm_s",
             "material_type", "oxygen_pct"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    ax = axes[i]
    m, b, r, p, _ = stats.linregress(df[feat], df[TARGET])
    ax.scatter(df[feat], df[TARGET], alpha=0.20, s=8, color=C_BLUE)
    xr = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(xr, m * xr + b, color=C_RED, lw=1.5, ls="--", label=f"r = {r:.3f}")
    ax.axhline(RA_SPEC, color=C_AMBER, lw=1.0, ls=":", alpha=0.8)
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel(TARGET, fontsize=9)
    ax.set_title(feat, fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)

plt.suptitle("Top Feature Correlations with Ra Roughness (amber dashed = spec limit)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


## 5 · Preprocessing

Ridge regression **requires standardisation** — the L2 penalty is scale-sensitive. Without scaling, variables measured in watts (∼2500) would be penalised far less than variables measured in millimetres (∼0.5), simply because of their unit. `StandardScaler` (zero mean, unit variance) ensures the penalty is applied equitably across all features.

This is the same requirement as Lasso (Project 09). The difference is that Ridge never zeros any coefficient — it shrinks all of them toward zero, keeping every feature in the model while dampening the instability caused by collinear predictors.


In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train : {X_train.shape[0]:,} rows")
print(f"Test  : {X_test.shape[0]:,} rows")
print(f"Scaling: StandardScaler (mean=0, std=1 per feature)")
print(f"\nFeature scales before standardisation (std):")
for feat, std in zip(FEATURES, scaler.scale_):
    print(f"  {feat:<26}: σ = {std:.3f}")
print("\n→ Without scaling, laser_power_w (σ ≈ 720) would dominate Ridge penalty")
print("  over focal_offset_mm (σ ≈ 0.6) — unfair penalisation.")


## 6 · Model Training

### Why Ridge for collinear laser process data?

When features are correlated — as `laser_power_w`, `cutting_speed_mm_s`, and `assist_gas_flow_l_min` clearly are — OLS spreads the shared variance across all correlated coefficients in an unstable way. A small change in the training set can flip a coefficient's sign or magnitude dramatically. This is not a numerical glitch; it is the mathematical consequence of near-singular design matrices.

Ridge adds an L2 penalty to the loss:

$$\text{Loss} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p}\beta_j^2$$

The effect: all coefficients shrink proportionally. For collinear features, Ridge distributes the shared effect *smoothly* across all correlated variables instead of assigning all explanatory power to one arbitrarily. The regularisation path — coefficient values across a range of α — makes this shrinkage visible and verifiable.

`RidgeCV` selects α via 5-fold cross-validation across 80 candidates on a log scale.


In [ ]:
# ── RidgeCV — find optimal alpha ─────────────────────────────────────
alphas_grid = np.logspace(-3, 3, 80)

ridge_cv = RidgeCV(alphas=alphas_grid, cv=5, scoring="r2")
ridge_cv.fit(X_train_sc, y_train)

best_alpha = ridge_cv.alpha_
print(f"RidgeCV — Optimal alpha: {best_alpha:.5f}")
print(f"  Evaluated {len(alphas_grid)} alpha candidates: [{alphas_grid[0]:.4f}, {alphas_grid[-1]:.0f}]")
print(f"  5-fold CV R²: {ridge_cv.score(X_train_sc, y_train):.4f}")


In [ ]:
# ── Ridge Regularisation Path ────────────────────────────────────────
alphas_path = np.logspace(-3, 3, 60)
coefs_path  = []

for a in alphas_path:
    r = Ridge(alpha=a)
    r.fit(X_train_sc, y_train)
    coefs_path.append(r.coef_)

coefs_path = np.array(coefs_path)

fig, ax = plt.subplots(figsize=(11, 5.5))
palette = [C_BLUE, C_RED, C_GREEN, C_AMBER, C_PURPLE, "#1ABC9C", "#E67E22", C_MUTED]
for i, (feat, color) in enumerate(zip(FEATURES, palette)):
    ax.plot(np.log10(alphas_path), coefs_path[:, i],
            lw=1.5, color=color, label=feat, alpha=0.85)

ax.axvline(np.log10(best_alpha), color="white", lw=2.0, ls="--",
           label=f"Optimal α = {best_alpha:.3f} (RidgeCV)")
ax.axhline(0, color=C_DARK, lw=0.5, ls=":")

ax.set_xlabel("log₁₀(α)  →  Higher α = stronger shrinkage toward zero", fontsize=11)
ax.set_ylabel("Standardised Coefficient", fontsize=11)
ax.set_title("Ridge Regularisation Path — All 8 Features\n"
             "Unlike Lasso, Ridge never zeroes coefficients — it shrinks all toward zero smoothly",
             fontsize=11, fontweight="bold")
ax.legend(loc="upper right", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print("Key observation from the regularisation path:")
print("  As α increases, collinear features (power, speed, gas) shrink together —")
print("  Ridge distributes their shared effect rather than amplifying one arbitrarily.")


In [ ]:
# ── Fit final Ridge + OLS for comparison ─────────────────────────────
ridge = Ridge(alpha=best_alpha)
ridge.fit(X_train_sc, y_train)
y_train_pred = ridge.predict(X_train_sc)
y_test_pred  = ridge.predict(X_test_sc)

ols = LinearRegression()
ols.fit(X_train_sc, y_train)
y_test_pred_ols = ols.predict(X_test_sc)

print(f"✓ Ridge (α = {best_alpha:.5f}) fitted")
print(f"✓ OLS (no regularisation) fitted for comparison")

# Coefficient comparison table
coef_compare = pd.DataFrame({
    "Feature"   : FEATURES,
    "Ridge_Coef": ridge.coef_,
    "OLS_Coef"  : ols.coef_,
    "Diff (OLS−Ridge)": ols.coef_ - ridge.coef_,
}).sort_values("Ridge_Coef", key=abs, ascending=False).reset_index(drop=True)

display(coef_compare.style.format({
    "Ridge_Coef": "{:+.5f}",
    "OLS_Coef"  : "{:+.5f}",
    "Diff (OLS−Ridge)": "{:+.6f}",
}).bar(subset=["Ridge_Coef", "OLS_Coef"], align="mid",
       color=[C_BLUE, C_AMBER]))

print("\nObservation: OLS assigns sharper extremes to collinear features.")
print("Ridge moderates those extremes — more stable for process guidance.")


## 7 · Model Evaluation

| Metric | Ridge | OLS | Operational Meaning |
|---|---|---|---|
| **R²** | **0.722** | 0.722 | 72.2% of Ra variance explained — similar accuracy, Ridge wins on stability |
| **RMSE** | **0.394 µm** | 0.394 µm | Prediction error relative to spec limit (3.2 µm) is 12% |
| **MAE** | **0.305 µm** | — | Typical absolute miss — adequate for go/no-go scheduling |

The practical story: Ridge and OLS deliver virtually identical test-set accuracy here. **This is expected and correct.** Ridge's advantage is not higher R² — it is coefficient stability under multicollinearity, which makes the model's guidance more trustworthy across the operating range. An OLS model would give unstable coefficient estimates for the collinear power–speed–gas triplet; Ridge gives interpretable, stable ones.


In [ ]:
r2_train = r2_score(y_train, y_train_pred)
r2_test  = r2_score(y_test, y_test_pred)
rmse_test= np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)
residuals= y_test.values - y_test_pred

print(f"Ridge Regression — Performance Summary")
print(f"  R² train:   {r2_train:.5f}")
print(f"  R² test:    {r2_test:.5f}")
print(f"  RMSE test:  {rmse_test:.5f} µm")
print(f"  MAE  test:  {mae_test:.5f} µm")
print(f"  OLS R² test:{r2_score(y_test, y_test_pred_ols):.5f}  (nearly identical)")
print(f"\n  RMSE / Spec limit ratio: {rmse_test / RA_SPEC * 100:.1f}%")
print(f"  → The model can usefully distinguish passes that will be near vs well inside spec.")

# Alpha sensitivity table
print("\nAlpha sensitivity:")
rows = []
for a_val in [0.01, 0.1, 1.0, best_alpha, 10.0, 100.0]:
    r = Ridge(alpha=a_val); r.fit(X_train_sc, y_train)
    r2t = r2_score(y_test, r.predict(X_test_sc))
    rows.append({"Alpha": round(a_val, 4), "R² Test": round(r2t, 5),
                 "Max |coef|": round(abs(r.coef_).max(), 5)})
display(pd.DataFrame(rows).style.highlight_max(subset=["R² Test"], color="#d5f5e3"))
print(f"\n✓ Selected alpha = {best_alpha:.5f}: cross-validated optimum.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── (A) Predicted vs Actual
ax = axes[0]
in_spec = y_test <= RA_SPEC
ax.scatter(y_test[in_spec],  y_test_pred[in_spec],
           alpha=0.50, s=12, color=C_BLUE,  label="Within spec")
ax.scatter(y_test[~in_spec], y_test_pred[~in_spec],
           alpha=0.50, s=12, color=C_RED,   label="Out of spec")
lims = [y_test.min() - 0.1, y_test.max() + 0.1]
ax.plot(lims, lims, color=C_DARK, ls="--", lw=1.5, label="Perfect prediction")
ax.axvline(RA_SPEC, color=C_AMBER, lw=1.2, ls=":", alpha=0.7)
ax.axhline(RA_SPEC, color=C_AMBER, lw=1.2, ls=":", alpha=0.7, label=f"Spec = {RA_SPEC} µm")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual Ra (µm)", fontsize=11)
ax.set_ylabel("Predicted Ra (µm)", fontsize=11)
ax.set_title(f"Predicted vs Actual — R² = {r2_test:.4f}", fontsize=12, fontweight="bold")
ax.legend(fontsize=8)

# ── (B) Residuals vs Fitted
ax2 = axes[1]
ax2.scatter(y_test_pred, residuals, alpha=0.35, s=10,
            c=[C_RED if abs(r) > 1.0 else C_BLUE for r in residuals])
ax2.axhline(0, color=C_DARK, lw=1.5, ls="--")
ax2.axhline(+2 * rmse_test, color=C_AMBER, lw=1, ls=":", label=f"±2·RMSE")
ax2.axhline(-2 * rmse_test, color=C_AMBER, lw=1, ls=":")
ax2.set_xlabel("Fitted Ra (µm)", fontsize=11)
ax2.set_ylabel("Residual (µm)", fontsize=11)
ax2.set_title("Residuals vs Fitted — Homoscedasticity Check", fontsize=12, fontweight="bold")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 8 · Interpretability

Ridge coefficients are standardised (one-standard-deviation units), making them directly comparable across features measured in different physical units. Unlike OLS coefficients under multicollinearity — which can be inflated, sign-flipped, or wildly unstable — Ridge coefficients represent **smooth, conservative estimates** of each variable's partial effect.

The most important interpretability story here is not a single number — it is the **coefficient stability comparison**: how differently OLS and Ridge assign importance to the collinear triplet (power, speed, gas).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# ── (A) Ridge coefficients
ax = axes[0]
coef_sorted = coef_compare.sort_values("Ridge_Coef", ascending=True)
bar_c = [C_BLUE if c < 0 else C_RED for c in coef_sorted["Ridge_Coef"]]
bars = ax.barh(coef_sorted["Feature"], coef_sorted["Ridge_Coef"],
               color=bar_c, alpha=0.82, edgecolor="none", height=0.60)
ax.axvline(0, color=C_DARK, lw=0.8)
for bar, val in zip(bars, coef_sorted["Ridge_Coef"]):
    offset = 0.003 if val >= 0 else -0.003
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f"{val:+.4f}", va="center",
            ha="left" if val >= 0 else "right", fontsize=9, color=C_DARK)
ax.set_xlabel("Standardised Ridge Coefficient")
ax.set_title(f"Ridge (α = {best_alpha:.3f}) Coefficients\n(stable under multicollinearity)",
             fontsize=11, fontweight="bold")

# ── (B) OLS vs Ridge — collinear triplet comparison
ax2 = axes[1]
triplet = ["laser_power_w", "cutting_speed_mm_s", "assist_gas_flow_l_min"]
idx = [FEATURES.index(f) for f in triplet]
x_pos = np.arange(len(triplet))
w = 0.35
ax2.bar(x_pos - w/2, [ols.coef_[i] for i in idx],
        width=w, color=C_AMBER, alpha=0.80, label="OLS (unstable)")
ax2.bar(x_pos + w/2, [ridge.coef_[i] for i in idx],
        width=w, color=C_BLUE, alpha=0.80, label="Ridge (stable)")
ax2.set_xticks(x_pos)
ax2.set_xticklabels(["laser_power", "cut_speed", "gas_flow"], fontsize=9)
ax2.axhline(0, color=C_DARK, lw=0.8)
ax2.set_ylabel("Standardised Coefficient")
ax2.set_title("Collinear Triplet: OLS vs Ridge\n"
              "Ridge distributes shared effect more evenly",
              fontsize=11, fontweight="bold")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Top drivers (Ridge coefficients, sorted by absolute impact):")
for _, row in coef_compare.iterrows():
    direction = "▲ More roughness" if row["Ridge_Coef"] > 0 else "▼ Less roughness"
    print(f"  {row['Feature']:<26}: {row['Ridge_Coef']:+.5f}  {direction}")


In [ ]:
# ── Permutation Importance — model-agnostic verification ─────────────
from sklearn.inspection import permutation_importance

pi = permutation_importance(ridge, X_test_sc, y_test,
                             n_repeats=30, random_state=42,
                             scoring="r2")

pi_df = pd.DataFrame({
    "Feature"   : FEATURES,
    "Importance": pi.importances_mean,
    "Std"       : pi.importances_std,
}).sort_values("Importance", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5))
bar_c = [C_RED if v > 0.01 else C_AMBER if v > 0.001 else C_MUTED for v in pi_df["Importance"]]
ax.barh(pi_df["Feature"][::-1], pi_df["Importance"][::-1],
        xerr=pi_df["Std"][::-1], color=bar_c[::-1],
        alpha=0.82, edgecolor="none", height=0.60, capsize=4)
ax.axvline(0, color=C_DARK, lw=0.8)
ax.set_xlabel("Mean R² drop when feature is permuted (higher = more important)", fontsize=10)
ax.set_title("Permutation Feature Importance — Ridge Model\n"
             "(error bars = ±1 std across 30 permutation repetitions)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

display(pi_df.style.format({"Importance": "{:.5f}", "Std": "{:.5f}"}))


## 9 · Statistical Validation

Ridge is a biased estimator — the L2 penalty deliberately introduces bias to reduce variance. Classical p-value inference does not apply directly to Ridge coefficients. However, residual diagnostics remain valid and important: they test whether the model has captured the data's structure adequately, regardless of the choice of regularisation method.


In [ ]:
from scipy.stats import normaltest

dag_stat, dag_p = normaltest(residuals)
print(f"D'Agostino-Pearson normality test (n={len(residuals)}):")
print(f"  Stat = {dag_stat:.4f}  |  p = {dag_p:.4f}")
print(f"  → {'Residuals normally distributed.' if dag_p > 0.05 else 'Mild non-normality — acceptable given n and RMSE.'}")

dw = durbin_watson(residuals)
print(f"\nDurbin-Watson: {dw:.4f}  (ideal ≈ 2)")
print(f"  → {'No autocorrelation.' if 1.5 < dw < 2.5 else 'Check record ordering.'}")

X_te_c = sm.add_constant(X_test_sc)
gq_stat, gq_p, _ = het_goldfeldquandt(residuals, X_te_c)
print(f"\nGoldfeld-Quandt homoscedasticity: stat = {gq_stat:.4f}  p = {gq_p:.4f}")
print(f"  → {'Constant variance — assumption holds.' if gq_p > 0.05 else 'Some heteroscedasticity — consider log transform of Ra.'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sns.histplot(residuals, bins=25, kde=True, color=C_BLUE, alpha=0.7, ax=ax)
ax.axvline(0, color=C_RED, ls="--", lw=1.5)
ax.set_xlabel("Residual (µm)"); ax.set_title("Residual Distribution", fontsize=12, fontweight="bold")

ax2 = axes[1]
(osm, osr), (slope, intercept, r) = probplot(residuals, dist="norm")
ax2.scatter(osm, osr, s=10, color=C_BLUE, alpha=0.65)
xl = np.linspace(min(osm), max(osm), 100)
ax2.plot(xl, slope * xl + intercept, color=C_RED, lw=1.5, ls="--")
ax2.set_xlabel("Theoretical Quantiles"); ax2.set_ylabel("Sample Quantiles")
ax2.set_title("Normal Q-Q Plot", fontsize=12, fontweight="bold")

plt.tight_layout(); plt.show()


## 10 · Process Simulator — Recipe Optimisation

Three scenarios show the dominant levers: **material type, thickness, and the power–speed–gas recipe**. The simulator reveals a critical insight: correcting an underpowered recipe on thick steel can recover over 1.2 µm of Ra — the difference between acceptable and out-of-spec cuts.


In [ ]:
def predict_roughness(laser_power_w, cutting_speed_mm_s, assist_gas_flow_l_min,
                       focal_offset_mm, material_thickness_mm, material_type,
                       oxygen_pct, shop_temp_c):
    """
    Predict surface roughness Ra for a given laser cutting recipe.

    Returns
    -------
    ra_um   : float — predicted Ra surface roughness (µm)
    status  : str   — 'Pass' or 'Fail' relative to spec limit
    margin  : float — distance to spec limit (µm); negative = outside spec
    """
    x_raw = pd.DataFrame([[laser_power_w, cutting_speed_mm_s, assist_gas_flow_l_min,
                            focal_offset_mm, material_thickness_mm, material_type,
                            oxygen_pct, shop_temp_c]], columns=FEATURES)
    ra_um  = ridge.predict(scaler.transform(x_raw))[0]
    status = "Pass ✓" if ra_um <= RA_SPEC else "Fail ✗"
    margin = RA_SPEC - ra_um
    return ra_um, status, margin


In [ ]:
# ══ Scenario A — Thin aluminium, high power, optimised recipe ═════════
a_ra, a_st, a_mg = predict_roughness(
    laser_power_w=3200, cutting_speed_mm_s=20, assist_gas_flow_l_min=10,
    focal_offset_mm=0.0, material_thickness_mm=3.0,
    material_type=1,  # aluminium
    oxygen_pct=8, shop_temp_c=23
)
print("═══ Scenario A — 3 mm Aluminium · High Power · Optimised ════════")
print(f"  Recipe: 3200W | 20 mm/s | 10 L/min gas | focal 0.0 | O₂ 8%")
print(f"  → Predicted Ra : {a_ra:.3f} µm   [{a_st}]")
print(f"  → Margin       : {a_mg:+.3f} µm to spec limit")

# ══ Scenario B — Thick steel, underpowered, too fast ══════════════════
b_ra, b_st, b_mg = predict_roughness(
    laser_power_w=2500, cutting_speed_mm_s=35, assist_gas_flow_l_min=5,
    focal_offset_mm=0.0, material_thickness_mm=12.0,
    material_type=0,  # steel
    oxygen_pct=25, shop_temp_c=23
)
print("\n═══ Scenario B — 12 mm Steel · Under-Powered · Too Fast ═════════")
print(f"  Recipe: 2500W | 35 mm/s | 5 L/min gas | focal 0.0 | O₂ 25%")
print(f"  → Predicted Ra : {b_ra:.3f} µm   [{b_st}]")
print(f"  → Margin       : {b_mg:+.3f} µm to spec limit")
print(f"  ⚠ Insufficient energy density for 12mm steel at this speed.")

# ══ Scenario C — Same thick steel, corrected recipe ═══════════════════
c_ra, c_st, c_mg = predict_roughness(
    laser_power_w=3800, cutting_speed_mm_s=18, assist_gas_flow_l_min=10,
    focal_offset_mm=0.0, material_thickness_mm=12.0,
    material_type=0,  # steel
    oxygen_pct=25, shop_temp_c=23
)
print("\n═══ Scenario C — 12 mm Steel · Corrected Recipe ═════════════════")
print(f"  Recipe: 3800W | 18 mm/s | 10 L/min gas | focal 0.0 | O₂ 25%")
print(f"  → Predicted Ra : {c_ra:.3f} µm   [{c_st}]")
print(f"  → Margin       : {c_mg:+.3f} µm to spec limit")
print(f"\n  ✅ Recipe correction (higher power, slower speed, more gas)")
print(f"     recovers {b_ra - c_ra:.3f} µm Ra — from {b_ra:.2f} to {c_ra:.2f} µm.")


In [ ]:
# ── 2D Contour: Power × Speed for 12mm Steel ─────────────────────────
power_range = np.linspace(1500, 4000, 60)
speed_range = np.linspace(5, 50, 60)
P, S = np.meshgrid(power_range, speed_range)

grid = pd.DataFrame({
    "laser_power_w"       : P.ravel(),
    "cutting_speed_mm_s"  : S.ravel(),
    "assist_gas_flow_l_min": 9.0,
    "focal_offset_mm"     : 0.0,
    "material_thickness_mm": 12.0,
    "material_type"       : 0,
    "oxygen_pct"          : 20.0,
    "shop_temp_c"         : 23.0,
})
Z = ridge.predict(scaler.transform(grid[FEATURES])).reshape(P.shape)

fig, ax = plt.subplots(figsize=(9, 6))
cf = ax.contourf(P, S, Z, levels=25, cmap="RdYlGn_r", alpha=0.88)
cbar = plt.colorbar(cf, ax=ax); cbar.set_label("Predicted Ra (µm)")
cs = ax.contour(P, S, Z, levels=[RA_SPEC], colors=["white"], linewidths=2.0)
ax.clabel(cs, fmt=f"Spec: {RA_SPEC} µm", fontsize=9, colors="white")
ax.contourf(P, S, Z, levels=[0, RA_SPEC], colors=["lime"], alpha=0.15, hatches=["////"])
ax.set_xlabel("Laser Power (W)", fontsize=11)
ax.set_ylabel("Cutting Speed (mm/s)", fontsize=11)
ax.set_title("Ra Response Surface — 12 mm Carbon Steel\n"
             "Green hatch = Power/Speed combinations that meet spec (Ra ≤ 3.2 µm)",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

print("Use this map as a process window reference for 12mm steel.")
print(f"Above the spec contour: reduce speed and/or increase power.")


## 11 · Final Reflection

---

### What Ridge revealed about this process

The laser cutting dataset has a structural collinearity problem that is not an artefact of data collection — it reflects real operating practice. Power, speed, and gas pressure move together because experienced operators have learned to increase them in concert. This is correct engineering practice, but it creates a near-singular design matrix that OLS cannot handle reliably.

Ridge regression resolves the problem without sacrificing any of the three variables: it shrinks all coefficients proportionally, distributing the shared explanatory power across the correlated triplet rather than amplifying one arbitrarily. The regularisation path — showing coefficient trajectories across 80 alpha values — makes this behaviour transparent and verifiable.

The model's hierarchy is physically grounded:

- **Material thickness (+0.385 per σ)** is the dominant driver — more material means more energy is required to achieve full-depth cuts, and when energy is insufficient, roughness increases.
- **Material type (−0.304 per σ)** reflects aluminium's superior thermal conductivity and molten material ejection characteristics — it consistently achieves lower Ra than carbon steel at equivalent energy settings.
- **Laser power (−0.267 per σ)** and **assist gas flow (−0.190 per σ)** together form the controllable energy delivery system. Ridge assigns stable, physically meaningful estimates to both despite their r = 0.797 correlation.

### The scenario result

Scenario B vs C is the process engineering payoff: on 12 mm carbon steel, an underpowered, over-speed recipe (2500W / 35 mm/s) predicts Ra = 4.17 µm — outside spec. Raising power to 3800W, reducing speed to 18 mm/s, and increasing gas flow to 10 L/min restores Ra to 2.94 µm — within spec, with 0.26 µm margin. The recipe change is quantified, not guessed.

### The Ridge vs Lasso choice

Both Ridge and Lasso regularise regression for correlated or high-dimensional problems. The choice depends on the domain:

- **Lasso** (Project 09, welding): appropriate when some features are genuinely irrelevant and feature elimination is valuable. Lasso identified 2 of 17 welding variables as zero.
- **Ridge** (this project): appropriate when all features have physical justification and the goal is stable coefficient estimation under collinearity. Ridge keeps all 8 laser parameters — because all 8 have legitimate engineering reasons for inclusion.

The regularisation path plot makes the key difference visible: Lasso paths reach exactly zero; Ridge paths asymptote toward zero but never cross it.

---

*LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa*
